# F1 Meets AI: 01 — Silverstone proof of concept

**Learning objectives:** trace a historical observation, convert it into a NumPy feature vector, understand PyTorch logits and binary loss, and check how a pit action changes the simulator.

This notebook is a guided experiment, not a final model. Run from the repository root. The saved replay is prior team evidence; synthetic tests are not race results. Live FastF1 and simulator cells are opt-in and were not executed in the authoring environment.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from f1_ai.pit_pipeline import FEATURES, encode, historical_examples, PitMLP, smoke_test, load_silverstone, run_race

ROOT = Path.cwd()
assert (ROOT / 'race-simulation').is_dir(), 'Start Jupyter from the repository root'
RUN_FASTF1 = False
RUN_SIMULATOR = False
OUT = ROOT / 'local_results'

## Part I — Historical replay (existing committed evidence)

**Question:** What does replaying the historical pit sequence establish? What does it not establish?

Inspect the existing comparison. A negative position difference means a better simulated finish. This is not a new run.

In [ ]:
replay = pd.read_csv(ROOT / 'race-simulation/experiments/sanity_Silverstone_2019.csv')
display(replay[['driver', 'real_pos', 'sim_pos', 'real_gap', 'sim_gap']])
print('Mean absolute position error:', (replay.sim_pos - replay.real_pos).abs().mean())
print('Rank correlation:', replay.sim_pos.corr(replay.real_pos, method='spearman'))
plt.figure(figsize=(5, 5))
plt.scatter(replay.real_pos, replay.sim_pos)
plt.plot([1, 20], [1, 20], linestyle='--')
plt.xlabel('Historical finishing position')
plt.ylabel('Simulated finishing position')
plt.title('Saved Silverstone 2019 replay')
plt.grid(alpha=.2)
plt.show()

## Part II — Build a common representation (NumPy)

Define the prediction time at the start of lap t. Features are race progress before lap t, previous-lap finishing position normalized by field size, and number of earlier pit stops. The target is whether the driver enters the pits during lap t.

We intentionally omit tyre age, compound, gaps and SC/VSC until source timing and encoding are aligned. A one-lap record cannot train a model that generalizes.

In [ ]:
x = encode(lap=2, total_laps=52, previous_position=4, previous_pits=0, field_size=20)
print(dict(zip(FEATURES, x)))
np.testing.assert_allclose(x, [1/52, 4/20, 0])
smoke_test()

**Inline question:** Why should changing the current lap's position or pit-in event not change the feature vector used to predict that same pit-in event? Why is a synthetic overfit test useful but not evidence of good predictions?

## Part III — PyTorch forward and backward pass

A three-feature MLP is deliberately small. The output is one logit per observation, not a probability. Sigmoid converts it to a probability; BCEWithLogitsLoss implements stable binary cross-entropy directly from logits.

In [ ]:
torch.manual_seed(7)
X = torch.tensor(np.stack([encode(t, 52, 4, 0, 20) for t in [2, 3]]))
y = torch.tensor([0., 1.])  # synthetic labels, not history
model = PitMLP()
logits = model(X)
loss = torch.nn.BCEWithLogitsLoss()(logits, y)
loss.backward()
print('X:', X.shape, 'logits:', logits.shape, 'loss:', loss.item())
print('probabilities:', logits.detach().sigmoid())
print('gradient shapes:', [tuple(p.grad.shape) for p in model.parameters()])

**Inline question:** What changes in the input shape if a GRU uses the last K laps instead of one observation? Why is a bidirectional LSTM inappropriate for a live decision when future laps are not available?

## Part IV — FastF1: real historical input (opt-in)

Set RUN_FASTF1=True after installing FastF1 and ensuring access to lap data. Do not silently replace failing downloads with synthetic observations. Record data access and missing rows. This is one driver/race for engineering only, not a train/test dataset.

In [ ]:
historical = None
if RUN_FASTF1:
    OUT.mkdir(exist_ok=True)
    historical = load_silverstone(OUT / 'fastf1_cache')
    historical.to_csv(OUT / 'historical_states.csv', index=False)
    display(historical.head())
    print('Pit labelled laps:', historical.loc[historical.pit == 1, 'lap'].tolist())
else:
    print('Real FastF1 execution skipped')

## Part V — TUM simulator: controlled intervention (opt-in)

Use the historical strategy as the comparison policy, then force one pit at lap 24 for VER only (compound A2), while other drivers retain their recorded strategies. This is a test action, not an optimized strategy. The wrapper observes the VSE inputs, modifies the returned action only for VER and checks actual recorded strategy info. The upstream simulator and original config are not modified.

**Acceptance:** two traces and a summary table; the alternative strategy has exactly one actual stop on lap 24. If the run fails, report the exception or timeout, not invented results.

In [ ]:
traces, outcomes = {}, []
if RUN_SIMULATOR:
    OUT.mkdir(exist_ok=True)
    for alternative in (False, True):
        trace, result = run_race(ROOT, alternative)
        traces[result['policy']] = trace
        outcomes.append(result)
        trace.to_csv(OUT / (result['policy'] + '_trace.csv'), index=False)
    comparison = pd.DataFrame(outcomes)
    comparison.to_csv(OUT / 'policy_comparison.csv', index=False)
    display(comparison)
    display(traces['fixed_lap_24'].query('22 <= lap <= 26'))
else:
    print('Live simulator execution skipped')

## Part VI — Connect historical and simulated observations

The two sources now share a schema. Matching feature names alone does not prove semantic equivalence. Verify chronology and the pit label, then compare position and stop-count differences during the historical replay. Position need not match because the simulator approximates the race. No trained NN controls the simulator yet.

In [ ]:
if historical is not None and 'historical' in traces:
    simulated = traces['historical'].copy()
    simulated['pit'] = simulated.action.notna().astype(int)
    joined = historical.merge(simulated[['lap', *FEATURES, 'pit']], on='lap',
                              suffixes=('_real', '_sim'), validate='one_to_one')
    display(joined.query('11 <= lap <= 24'))
    print('Pit label disagreements:', joined.query('pit_real != pit_sim').lap.tolist())
    joined.to_csv(OUT / 'feature_comparison.csv', index=False)
else:
    print('Enable both live steps to compare actual features')

## Part VII — Research discussion and next experiments

1. Explain exactly what was changed between the two race runs.
2. Check whether all 52 laps were completed before comparing times.
3. If the alternative is faster, what assumptions could invalidate that finding?
4. Specify the next data audit before training on multiple whole races.
5. Do not infer predictive accuracy from the toy overfit or train on the proof-of-concept evaluation race.

See the unassigned workstreams in GitHub and the experiment plan for deliverables.